# Exercices XP : MCP minimal sur STDIO (Étudiant)

Construisez un petit serveur et client MCP communiquant via STDIO. Ce code doit être exécuté dans un notebook Jupyter local, pas dans Colab.

## Ce que vous allez apprendre
- Comment MCP structure les hosts/clients/serveurs et pourquoi STDIO est idéal localement.
- Comment enregistrer un outil (action) et une ressource (contexte en lecture seule) sur un serveur.
- Comment écrire un client qui initialise, liste et invoque ces fonctionnalités.

## Installation
Exécutez la cellule d'installation, puis redémarrez le runtime si Colab le demande. Python 3.10+ requis.

In [ ]:
# Étape 1 : Installer MCP CLI + SDK
%pip install -qU "mcp[cli]"

In [ ]:
# Étape 2 : Vérification rapide
!python --version
!mcp --help | head -n 5

## A. Serveur (server.py)
Créez un petit serveur MCP nommé "Demo" avec :
- Un outil `add(a: int, b: int) -> int` qui retourne la somme.
- Un modèle de ressource `greeting://{name}` qui retourne "Bonjour, {name} !".
- Démarrez la boucle STDIO dans `__main__`.

In [ ]:
%%writefile server.py
# Importation de FastMCP pour créer le serveur MCP
from mcp.server.fastmcp import FastMCP

# Création de l'instance du serveur MCP nommée "Demo"
mcp = FastMCP("Demo")


# Étape A1 : Déclaration de l'outil 'add' qui additionne deux entiers
@mcp.tool()
def add(a: int, b: int) -> int:
    """Retourne la somme de deux entiers."""
    # TODO: retourner la somme de a et b
    return a + b


# Étape A2 : Déclaration de la resource template 'greeting' qui salue un nom
@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Retourne un message de salutation pour le nom donné."""
    # TODO: retourner la chaîne formatée "Bonjour, {name} !"
    return f"Bonjour, {name} !"


# Étape A3 : Lancement du serveur sur STDIO si le fichier est exécuté directement
if __name__ == "__main__":
    # TODO: démarrer la boucle du serveur sur STDIO
    mcp.run(transport="stdio")

## B. Client (client.py)
Écrivez un client qui :
1) Lance le serveur via STDIO en utilisant le CLI MCP.
2) Initialise une session.
3) Liste les ressources et les outils, affiche leurs noms.
4) Lit `greeting://hello` et affiche le contenu.
5) Appelle l'outil `add` avec a=1, b=7 et affiche le résultat.

In [ ]:
%%writefile client.py
# Importation des modules nécessaires pour le client MCP
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Étape B1 : Paramètres du serveur STDIO (CLI MCP lance server.py)
server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)


# Fonction utilitaire pour extraire le texte des réponses MCP
def extract_content(payload):
    """Meilleur effort pour extraire le texte des réponses MCP."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        content = payload.content
        if isinstance(content, list) and content:
            first = content[0]
            if hasattr(first, "text"):
                first.text
            return str(first)
        return str(content)
    return str(payload)


# Étape B2 : Logique principale du client
async def run():
    # Connexion STDIO au serveur
    async with stdio_client(server_params) as (read, write):
        # Création de la session MCP
        async with ClientSession(read, write) as session:
            # Initialisation de la session
            await session.initialize()

            # TODO: lister les ressources et afficher leurs URI
            resources = await session.list_resources()
            print("Ressources :")
            for resource in resources.resources:
                print(f"- {resource.uri}")

            # greeting://{name} est enregistré comme un TEMPLATE de ressource
            templates = await session.list_resource_templates()
            print("\nTemplates de ressources :")
            for tmpl in templates.resourceTemplates:
                print(f"- {tmpl.uriTemplate}")

            # TODO: lister les outils et afficher leurs noms
            tools = await session.list_tools()
            print("\nOutils :")
            for tool in tools.tools:
                print(f"- {tool.name}")

            # TODO: lire greeting://hello et afficher le contenu
            greeting_payload = await session.read_resource("greeting://hello")
            print(f"\nSalutation : {extract_content(greeting_payload)}")

            # TODO: appeler add avec a=1, b=7 et afficher le résultat
            add_result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print(f"\nRésultat addition : {extract_content(add_result)}")


# Étape B3 : Point d'entrée du script client
if __name__ == "__main__":
    asyncio.run(run())

## C. Exécution
Dans un terminal (le client lance le serveur) :
```
python client.py
```

Ou dans deux terminaux séparés :
```
mcp run server.py
python client.py
```

Dans Colab, exécutez la cellule suivante (le client lancera le serveur automatiquement).

In [ ]:
# Étape de lancement : exécuter le client (qui lance le serveur sur STDIO)
!python client.py

## Dépannage
- `mcp: command not found` ? Réexécutez la cellule d'installation ou redémarrez le runtime.
- `Connection closed` ? Ouvrez un second terminal et lancez `mcp run server.py` pour voir les erreurs du serveur.
- Erreurs de type ? Assurez-vous que les arguments JSON sont des entiers pour `add`.